In [1]:
from maite_datasets.object_detection import SeaDrone

# Downloads to ./data/seadrone on first run (~1.2 GB for the validation split).
seadrone = SeaDrone(root="./data", image_set="val", download=True)
print(f"{len(seadrone)} images")

/builds/jatic/aria/dataeval-flow/.nox/docs/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1547 images


In [2]:
from dataeval_flow.config import (
    DatasetProtocolConfig,
    PipelineConfig,
    SourceConfig,
    ViewConfig,
    ViewOperation,
)
from dataeval_flow.config.schemas import (
    MetadataTriageTaskConfig,
    MetadataTriageWorkflowConfig,
)
from dataeval_flow.workflow import run_task

triage_workflow = MetadataTriageWorkflowConfig(
    name="triage",
    max_examples=6,  # distinct values shown per column in the report
)

task = MetadataTriageTaskConfig(
    name="triage_seadrone",
    workflow="triage",
    sources="seadrone_val",
)

config = PipelineConfig(
    datasets=[DatasetProtocolConfig(name="seadrone", dataset=seadrone)],
    views=[
        ViewConfig(
            name="sample",
            operations=[
                ViewOperation(type="Shuffle", params={"seed": 0}),
                ViewOperation(type="Limit", params={"size": 200}),
            ],
        )
    ],
    sources=[SourceConfig(name="seadrone_val", dataset="seadrone", view="sample")],
    workflows=[triage_workflow],
    tasks=[task],
)

In [3]:
result = run_task(task, config)
print(f"success={result.success}  health={result.health['status']}")

success=True  health=warning


/builds/jatic/aria/dataeval-flow/src/dataeval_flow/binning.py:438: UserWarning: `date_time` was dropped: nearly every row holds a different value, so the column identifies rows rather than grouping them, and it is not numeric so there is no order along which to cut it into groups. Map the values onto a smaller vocabulary to keep the factor. See Metadata.dropped_factors.
  factor_info = metadata.factor_info
/builds/jatic/aria/dataeval-flow/src/dataeval_flow/binning.py:438: UserWarning: `altitude`, `compass_heading`, `frame`, `gimbal_heading`, `gimbal_pitch`, `object_id`, `object_size`, `speed`, `xspeed` and `yspeed` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"altitude": [...]} to control this.
  factor_info = metadata.factor_info
/builds/jatic/aria/dataeval-flow/src/dataeval_flow/

In [4]:
print(result.report())


  15 FACTORS, 24 FINDINGS (3 BLOCKING).
  Timestamp:    2026-09-15T18:05:54.993545+00:00
  Duration:     14.21s
  Source:       seadrone_val (seadrone[sample])
--------------------------------------------------------------------------------

  SUMMARY
  -------
  Unreadable factors ......................................... 3 factors  [!!]
  Columns dominated by one value ............................. 5 factors  [..]
  Degenerate factors ......................................... 1 factors  [..]
  Unpinned continuous bins .................................. 10 factors  [..]
  Unpinned categorical vocabularies .......................... 5 factors  [..]
  Suggested policy .............. add to configuration under `metadata:`  [..]
  Verified ................................................. 5 recovered  [..]

  Health: 1 warning(s) [!!] — review flagged findings

  UNREADABLE FACTORS                                                   3 factors

  [blocking] date_time [cardinality_over_budge

In [5]:
latitude = next(f for f in result.data.raw.findings if f.factor == "latitude")

print("category :", latitude.category)
print("severity :", latitude.severity)
print("reasons  :", latitude.reasons)
print("suggested:", latitude.suggestion.corrections)
print("runnable :", latitude.suggestion.complete)

category : unreadable
severity : blocking
reasons  : ('mixed_types',)
suggested: [{'kind': 'remap', 'factor': 'latitude', 'rules': [{'match': 'N', 'to': None}]}]
runnable : False


In [6]:
print(result.data.raw.suggested_policy_yaml)

metadata:
- name: standard
  corrections:
  - kind: parse_datetime
    factor: date_time
    every: day
  - kind: remap
    factor: latitude
    rules:
    - match: N
      to: null        # TODO
  - kind: remap
    factor: longitude
    rules:
    - match: E
      to: null        # TODO
  - kind: remap
    factor: altitude
    rules:
    - match: -1.0
      to: null        # TODO
  - kind: remap
    factor: compass_heading
    rules:
    - match: -1.0
      to: null        # TODO
  - kind: remap
    factor: gimbal_heading
    rules:
    - match: -1.0
      to: null        # TODO
  - kind: remap
    factor: gimbal_pitch
    rules:
    - match: -1.0
      to: null        # TODO
  - kind: remap
    factor: speed
    rules:
    - match: -1.0
      to: null        # TODO
  exclude:
  - object_id
  continuous_factor_bins:
    frame: 5
    object_size: 12
    xspeed: 2
    yspeed: 2



In [7]:
from dataeval_flow.config.schemas import MetadataPolicyConfig

policy = MetadataPolicyConfig.model_validate(
    {
        "name": "seadrone",
        "corrections": [
            {"kind": "parse_datetime", "factor": "date_time", "every": "day"},
            # Map hemisphere letters to NaN (missing values)
            {"kind": "remap", "factor": "latitude", "rules": [{"match": "N", "to": float("nan")}]},
            {"kind": "remap", "factor": "longitude", "rules": [{"match": "E", "to": float("nan")}]},
            # Map -1.0 sentinel values to NaN
            *(
                {"kind": "remap", "factor": name, "rules": [{"match": -1.0, "to": float("nan")}]}
                for name in ("altitude", "compass_heading", "gimbal_heading", "gimbal_pitch", "speed")
            ),
        ],
        # Exclude object identifier
        "exclude": ["object_id"],
        "continuous_factor_bins": result.data.raw.suggested_policy["continuous_factor_bins"],
    }
)

corrected = config.model_copy(
    update={
        "metadata": [policy],
        "workflows": [triage_workflow.model_copy(update={"metadata": "seadrone"})],
    }
)
result2 = run_task(task, corrected)
print(f"success={result2.success}  health={result2.health['status']}")

success=True  health=ok


/builds/jatic/aria/dataeval-flow/src/dataeval_flow/binning.py:438: UserWarning: `altitude`, `compass_heading`, `gimbal_heading`, `gimbal_pitch`, `latitude`, `longitude` and `speed` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"altitude": [...]} to control this.
  factor_info = metadata.factor_info
/builds/jatic/aria/dataeval-flow/src/dataeval_flow/binning.py:438: UserWarning: Declared cuts left bins unused: frame (4 of 5 bins hold rows), object_size (9 of 12 bins hold rows). The encoding still applies and the codes are unchanged; this is the data no longer matching the policy, not an error. Call reencode() to refit — which moves codes — or leave it and read the empty groups as the finding they are.
  factor_info = metadata.factor_info


In [8]:
before = result.data.raw
after = result2.data.raw
print(f"factors : {before.factor_count} -> {after.factor_count}")
print(f"findings: {len(before.findings)} -> {len(after.findings)}")
print(f"blocking: {result.metadata.blocking} -> {result2.metadata.blocking}")

factors : 15 -> 17
findings: 24 -> 21
blocking: 3 -> 0


In [9]:
json_str = result.export(fmt="json")
print(f"JSON output: {len(json_str)} characters")

JSON output: 172304 characters
